[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/skinning/skinning.ipynb)

# Skinning: Blend the Motions, or Blend the Maps

This example bends a skin over two bones, the way an animated character's arm is skinned. Each point of the skin follows both bones, weighted by how near it lies to each, and the question is how to mix two motions. Mixing where the points would go, the usual way, makes a twisted limb collapse in the middle, like a wrung-out towel; mixing the motions themselves keeps every point at its distance from the bone.

A bone's motion is a motor, and its action on points is the map `motor >> Point`. Blending the maps and applying the blend is the usual linear blend skinning; blending the motors, normalising, and applying the sandwich keeps the result a rigid motion. In numga both are one line, the maps and the motors being objects of the same kind, and the collapse of the first is one number: the radius of the skin about the bone.

In the matrix notation of computer graphics the point map reads as a $4\times4$ matrix and the usual blend as a weighted sum of matrices. In dual quaternion notation a motor reads as a unit dual quaternion, and motor blending as dual quaternion blending.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from examples.animation import save_figure
from examples.surfaces.skinning import render
from examples.surfaces.skinning.core import Motor, Point, Scalar, cylinder, mv, radius

np.set_printoptions(precision=4, suppress=True)

## 1. A cylinder on two bones

The skin is a unit cylinder about the x axis, from 0 to 1, each ring weighted towards the second bone by its distance along. The first bone stays; the second is twisted about the shared axis.

In [ ]:
# The cylinder's rings along the bones and vertices around each, and how far the second bone is twisted.
n_rings, n_around = 12, 24
twist_angle = np.radians(150.0)

In [ ]:
skin, weight = cylinder(n_rings, n_around)                                     # [vertices] Point, Scalar
root = mv.rotor()                                                              # [] Motor
twist = (mv.yz * (twist_angle / 2)).exp()                                      # [] Motor

## 2. Three blends

Blending the motors and normalising keeps a rigid motion at every weight; the slerp, the exponential of the weighted logarithm of the relative motor, follows the geodesic between them and spaces the twist evenly; blending the maps averages two points of a circle and pulls them in towards the chord. From left to right, the three skins: blended motors, slerp, blended maps.

In [ ]:
def blend_skin(vertices: Point, weight: Scalar, root: Motor, tip: Motor):
    """Deform vertices by normalised motor blending, motor slerp, and map blending."""
    # Blend motors and normalise, or blend their point maps directly.
    motor_skin = (root + weight * (tip - root)).normalized() >> vertices
    matrix_skin = ((root >> Point) + weight * ((tip >> Point) - (root >> Point)))(vertices)

    # Slerp follows the geodesic: the log of the relative motor, scaled by the weight and
    # exponentiated. The normalised lerp is a chord of it, so both are rigid but their twist
    # angles are spaced differently along the blend.
    slerp_skin = ((tip / root).log() * weight).exp() * root >> vertices
    return motor_skin, slerp_skin, matrix_skin


motor_skin, slerp_skin, matrix_skin = blend_skin(skin, weight, root, twist)    # [vertices] Point each
save_figure(render.draw_skinning(motor_skin, slerp_skin, matrix_skin, n_rings, n_around), "skinning");

In [ ]:
print("smallest radius about the bone, blended motors:", radius(motor_skin).to_array().min())
print("smallest radius about the bone, blended maps:  ", radius(matrix_skin).to_array().min(), " cosine of half the twist:", np.cos(twist_angle / 2))

In [ ]:
# checks
# Both motor blends keep every vertex at unit radius; the map blend pulls a vertex of weight w in to the
# chord between its two images.
w = weight.to_array()
np.testing.assert_allclose(radius(motor_skin).to_array(), 1.0, atol=1e-12)
np.testing.assert_allclose(radius(slerp_skin).to_array(), 1.0, atol=1e-12)
np.testing.assert_allclose(radius(matrix_skin).to_array(), np.sqrt(1 - 2 * w * (1 - w) * (1 - np.cos(twist_angle))), atol=1e-12)